In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.listdir()

In [ ]:
%cd "/content/drive/My Drive/kaggle_rsna_problem"

In [ ]:
os.listdir()

In [ ]:
# # About the dataset from kaggle
# kaggle_rsna_problem/
# │
# ├── train.csv
# ├── train_localizers.csv
# ├── series/                  ← contains DICOM folders per series ID
# ├── segmentations/          ← contains NIfTI vessel segmentations

In [ ]:
import pandas as pd
df_train=pd.read_csv('train.csv')

In [ ]:
import os
import pydicom
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
import argparse

# ------------ CONFIGURATION ------------
SOURCE_DIR = "series"
TARGET_DIR = "preprocessed_basic/series"
CHECKPOINT_CSV = "preprocessed_basic/series_checkpoint.csv"
TARGET_SHAPE = (128, 128, 128)  # (Z, Y, X)
# --------------------------------------

def load_dicom_series(series_path):
    dicoms = []
    for fname in sorted(os.listdir(series_path)):
        path = os.path.join(series_path, fname)
        if fname.lower().endswith(".dcm"):
            ds = pydicom.dcmread(path)
            dicoms.append(ds)

    dicoms.sort(key=lambda d: float(d.ImagePositionPatient[2]) if hasattr(d, 'ImagePositionPatient') else 0)
    images = np.stack([d.pixel_array.astype(np.float32) for d in dicoms])

    # Normalize pixel values to [0, 1]
    images -= np.min(images)
    if np.max(images) != 0:
        images /= np.max(images)

    return images

def resize_volume(volume, target_shape):
    # Resize to target shape using interpolation
    # Assume (Z, Y, X)
    current_shape = volume.shape
    resized = cv2.resize(volume.transpose(1,2,0), (target_shape[2], target_shape[1]))  # Resize in X,Y
    resized = resized.transpose(2,0,1)
    final = np.zeros(target_shape, dtype=np.float32)
    z_rescale = cv2.resize(resized[:, :, :], (target_shape[2], target_shape[0]))
    return cv2.resize(volume, dsize=(target_shape[2], target_shape[1]))

def resize_3d(volume, target_shape):
    from scipy.ndimage import zoom
    factors = (
        target_shape[0] / volume.shape[0],
        target_shape[1] / volume.shape[1],
        target_shape[2] / volume.shape[2],
    )
    return zoom(volume, zoom=factors, order=1)  # linear interpolation

def save_checkpoint(series_id, status):
    if not os.path.exists(CHECKPOINT_CSV):
        df = pd.DataFrame(columns=["SeriesInstanceUID", "Status"])
        df.to_csv(CHECKPOINT_CSV, index=False)

    df = pd.read_csv(CHECKPOINT_CSV)
    if series_id in df['SeriesInstanceUID'].values:
        df.loc[df['SeriesInstanceUID'] == series_id, 'Status'] = status
    else:
        df = df.append({"SeriesInstanceUID": series_id, "Status": status}, ignore_index=True)

    df.to_csv(CHECKPOINT_CSV, index=False)

def preprocess_series():
    os.makedirs(TARGET_DIR, exist_ok=True)
    all_series = sorted(os.listdir(SOURCE_DIR))
    done_series = set()

    if os.path.exists(CHECKPOINT_CSV):
        done_series = set(pd.read_csv(CHECKPOINT_CSV)['SeriesInstanceUID'].values)

    for series_id in tqdm(all_series, desc="Preprocessing Series"):
        series_path = os.path.join(SOURCE_DIR, series_id)
        target_path = os.path.join(TARGET_DIR, f"{series_id}.npy")

        if series_id in done_series:
            continue

        try:
            volume = load_dicom_series(series_path)
            volume = resize_3d(volume, TARGET_SHAPE)
            np.save(target_path, volume)
            save_checkpoint(series_id, "Done")
        except Exception as e:
            print(f"Failed on {series_id}: {str(e)}")
            save_checkpoint(series_id, "Failed")

if __name__ == "__main__":
    preprocess_series()
